# Electrostatics: Poisson Equation for a Point Charge with Robin Boundary Condition

This notebook follows on from **E-Statics-PC-Neumann-Dirichlet**, which solved the
potential of a point charge with homogeneous Neumann and Dirichlet conditions. Both
were shown to disagree with the analytical solution as the boundary is approached —
Dirichlet forces $\Phi = 0$ at a finite radius, and Neumann settles onto a plateau.

Here a third option is used. A Robin condition prescribes the *decay* of the
potential at the boundary rather than its value or its flux,

$$\frac{\partial \Phi}{\partial n} = -\frac{\Phi}{r},$$

which is exactly the behaviour of a point charge in free space. The boundary
therefore no longer contradicts the physics it truncates, and the numerical result
stays close to $1/(4\pi\varepsilon r)$ across the whole domain.

On a **spherical** boundary centred on the charge this condition would be exact: the
outward normal coincides everywhere with the radial direction, so $\partial\Phi/\partial n$
*is* $\partial\Phi/\partial r$, and the truncation introduces no error at all. On the
cubic grid used here the two directions agree only at the centres of the six faces
and diverge towards the edges and corners, where the normal is furthest from radial.
The residual error is therefore geometric rather than a shortcoming of the condition
itself, and it is largest along the diagonals — which is where the comparison below
samples.

A second, complementary route to accuracy is simply to move the boundary further
away. Any boundary condition improves as $r \to \infty$ — Dirichlet converges to the
correct free-space solution once $\Phi \to 0$ is imposed where the potential is
genuinely negligible. The drawback is cost: enlarging a uniform grid raises the
number of degrees of freedom with the cube of the extent.

Non-equidistant grids avoid most of that cost. Because the field far from the source
varies slowly, the added cells do not need the resolution of the region of interest,
and a few geometrically growing cells extend the domain substantially for a modest
increase in unknowns. The final section combines both ideas: Robin conditions on a
grid extended by graded outer cells.

Identical code to **E-Statics-PC-Neumann-Dirichlet**:

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..")) 
Pkg.instantiate() 
using FITToolbox;
using CairoMakie;
using LinearAlgebra;
using SparseArrays;

x_size, y_size, z_size = 100,100,100
resolution = 2

MyDomain = create_domain([x_size,y_size,z_size],[resolution,resolution,resolution];units="m",σ=0,ε_r=1,μ_r=1); 
p_q = get_index_entity(MyDomain, DualVolume(), x_size/2.0, y_size/2.0, z_size/2.0; units="m", atol=1e-9)

q = zeros(Float64, MyDomain.Np);
q[p_q] = 1.602e-19; #C
Mε = get_permittivity(MyDomain)
dead_edges = get_ghost_matrix(MyDomain)

G = dead_edges * get_gradient(MyDomain, Primal())
L = G' * Mε * G;

**Assembling the Robin term.** The Robin condition has to be turned into a matrix
that can be added to the system. Every primal node $p$ sits at the centre of a dual
volume bounded by six dual facets, and Gauss's law over that volume reads

$$\sum_{\text{6 facets}} \hat{\hat{d}} = q_p .$$

For an interior node all six facets are shared with a neighbouring dual volume, and
$\tilde{\mathbf{S}}\mathbf{M}_\varepsilon\mathbf{G}$ already accounts for every one
of them. For a node on the outer shell some facets face out of the domain instead,
and the flux through those is simply missing from the system — there is no
neighbour to supply it.

The Robin condition fills that gap. With $\partial\Phi/\partial n = -\Phi/r$ the
outward flux through an exterior facet of area $\tilde{A}_p$ is

$$\hat{\hat{d}}_{\text{ext}} = \varepsilon_0\, \tilde{A}_p\, \frac{\Phi_p}{r_p} ,$$

which depends only on the potential at that one node. It therefore contributes a
single diagonal entry, and the entire boundary condition is a diagonal matrix
*added* to the system rather than substituted into it:

$$(\mathbf{L} + \mathbf{R})\,\Phi = q, \qquad
  \mathbf{R}_{pp} = \frac{\varepsilon_0\, \tilde{A}_p}{r_p} .$$

Two consequences follow from it being an addition. The source $q$ is used
unmodified, since no equation has been deleted and Gauss's law still holds at every
node. And the added positive diagonal removes the null space of $\mathbf{L}$, so
unlike the Neumann case no node has to be pinned to fix the gauge.

**Which facets are exterior.** The loop skips any node whose three indices are all
interior, since those carry no Robin term. For the rest, how many of the six dual
facets face outward depends on how many grid faces the node borders: one for a node
in the middle of a face, two along an edge, three at a corner. The three index tests
identify exactly those cases, and accumulating a contribution as each test passes
handles all of them without special-casing.

Note that $i = 1$ and $i = N_u$ are opposite sides of the domain but share a single
test. That works because a dual facet's area depends only on the two dual edges
tangent to it: the exterior facet at either end of the $u$-axis is spanned by
$\tilde{v}_j$ and $\tilde{w}_k$ and has the same area. Hence `Ãu[j,k]` in both
cases, and correspondingly `Ãv[i,k]` and `Ãw[i,j]` for the other two axes — each
array indexed by the directions tangent to its facet, not by the facet's own normal.

**The radius.** Distances are measured from the source position supplied by the
caller, not from the geometric centre of the grid: the $1/r$ decay is a statement
about distance from the charge, and the two coincide only when the charge happens to
be centred. Node coordinates are read through `get_position_of_index`, the same
lookup table `get_index_entity` uses, so the boundary term cannot drift out of step
with the rest of the package. The one-argument method defaults the source to the
centre of the domain for the common case.

**Sparsity.** Only the shell has nonzero entries — roughly $6N^2$ out of $N^3$ — so
the matrix is assembled from the index list rather than as a full diagonal. On a
$51^3$ grid that is about 15,000 stored values instead of 132,651.

**Assumptions.** On a *spherical* boundary centred on the charge this condition
would be exact: the outward normal is everywhere radial, so $\partial\Phi/\partial n$
is $\partial\Phi/\partial r$ and the truncation introduces no error. On a cubic grid
the two directions agree only at the centres of the six faces and diverge towards
the edges and corners, so a purely geometric error remains, largest along the
diagonals. The $1/r$ law additionally assumes a net charge; a neutral configuration
has a dipole far field decaying as $1/r^2$ and would need $2/r$ in its place.

In [ ]:
function get_robin_matrix(config::FITDomain, xc, yc, zc; units="m")
    ε₀ = 8.8541878188e-12   # F/m

    xc, yc, zc = FITToolbox.convert_to_meter(xc, yc, zc, units)
    Nu, Nv, Nw, Np = config.Nu, config.Nv, config.Nw, config.Np
    Ãu, Ãv, Ãw = config.dual_facets_u, config.dual_facets_v, config.dual_facets_w

    Ã = zeros(Np)   # exterior dual facet area attached to each boundary node
    r = zeros(Np)   # distance from that node to the source

    for k in 1:Nw, j in 1:Nv, i in 1:Nu
        # only the skin carries a Robin term; skip the interior
        (i in (1,Nu) || j in (1,Nv) || k in (1,Nw)) || continue
        p = i + (j-1)*Nu + (k-1)*Nu*Nv

        x, y, z = get_position_of_index(config, PrimalNode(), X(), i, j, k)
        r[p] = hypot(x - xc, y - yc, z - zc)

        # a boundary node's dual volume has one exterior facet per grid face it
        # borders — one on a face, two on an edge, three at a corner
        i in (1, Nu) && (Ã[p] += Ãu[j, k])
        j in (1, Nv) && (Ã[p] += Ãv[i, k])
        k in (1, Nw) && (Ã[p] += Ãw[i, j])
    end

    # ∂Φ/∂n = -Φ/r for a 1/r far field, so each node contributes ε₀·Ã/r
    idx = findall(!iszero, Ã)
    return sparse(idx, idx, ε₀ .* Ã[idx] ./ r[idx], Np, Np)
end

# default: source at the centre of the domain
get_robin_matrix(config::FITDomain) = get_robin_matrix(config,
    (config.nodes_u[1] + config.nodes_u[end])/2,
    (config.nodes_v[1] + config.nodes_v[end])/2,
    (config.nodes_w[1] + config.nodes_w[end])/2)

#Apply the boundary condition
Rob = get_robin_matrix(MyDomain, x_size/2, y_size/2, z_size/2; units="m")
Lbc = L + Rob
#Solve
Φ_Robin = cholesky(Lbc; check=true) \ q;

In [ ]:
plot_nodal_values(MyDomain, Primal(), Φ_Robin, Z();
                  pos = z_size/2.0, units="m",
                  clip_range = (1e-12, 1e-8),
                  logscale = true,
                  plot_negative_gradient = true)

**Extending the domain.** The Robin condition improves as the boundary moves further
from the source, since the $1/r$ law it prescribes becomes an ever better description
of the true field there. Enlarging a uniform grid is expensive — the number of
unknowns grows with the cube of the extent — but the added region does not need the
resolution of the region of interest: the field far from the charge varies slowly,
and a coarse cell resolves it perfectly well.

The `create_domain(Edges_U, Edges_V, Edges_W)` constructor takes the cell widths
directly, so a graded grid is a matter of prepending and appending a few growing
cells to the uniform core:

```julia
edges = vcat(20, 13, 9, 6, 3.5, fill(float(resolution), MyDomain.Nu - 1), 3.5, 6, 9, 13, 20)
```

The widths grow by a factor of roughly $1.5$ from one cell to the next, starting at
$3.5\;\text{m}$ next to the $2\;\text{m}$ core and reaching $20\;\text{m}$ at the
outer face. Keeping the ratio moderate matters: an abrupt jump in cell size degrades
the accuracy of the discretisation at the interface, so the grading is spread over
five cells rather than made in one step.

Five cells per side extend the domain from $100\;\text{m}$ to $203\;\text{m}$ while
raising the node count only from $51$ to $61$ per direction. The boundary moves
$51.5\;\text{m}$ further out for about $70\%$ more unknowns, against the eightfold
increase a uniform grid of the same extent would cost.

**Restricting the result.** The enlarged grid has its own indexing, so its solution
cannot be compared directly against the earlier ones. Because the core is built with
exactly `MyDomain`'s cells at the same resolution, however, the two grids coincide
node for node in the physical region, offset by the number of added cells on each
side. Reshaping the solution to three dimensions and slicing out that window returns
a vector with `MyDomain`'s length and indexing, so the same sampling line, radii and
analytical reference apply to it as to every other result.

The offset is derived from the two grids rather than written by hand: an error of one
cell would shift the curve by $2\;\text{m}$, which on a logarithmic axis reads as a
modest inaccuracy rather than as the bug it is.

In [ ]:
edges = vcat(20,13,9,6,3.5, fill(float(resolution), MyDomain.Nu - 1), 3.5,6,9,13,20)

Enlargedomain = create_domain(edges, edges, edges; units="m", σ=0, ε_r=1, μ_r=1)

g_centre = ntuple(d -> Enlargedomain.nodes_u[end]/2, 3)
p_q = get_index_entity(Enlargedomain, DualVolume(), g_centre... ; units="m", atol=1e-9)

q = zeros(Float64, Enlargedomain.Np);
q[p_q] = 1.602e-19; #C


Mε = get_permittivity(Enlargedomain)
dead_edges = get_ghost_matrix(Enlargedomain)

G = dead_edges * get_gradient(Enlargedomain, Primal())
L = G' * Mε * G

Rob = get_robin_matrix(Enlargedomain, g_centre...; units="m")
Lbc = L + Rob
Φ_Robin_Enlarged = cholesky(Lbc; check=true) \ q;

n_extra = (Enlargedomain.Nu - MyDomain.Nu) ÷ 2 #number of additional nodes on each side
core = n_extra+1 : n_extra+MyDomain.Nu
Φ_Robin_Enlarged = vec(reshape(Φ_Robin_Enlarged, Enlargedomain.Nu, Enlargedomain.Nv, Enlargedomain.Nw)[core, core, core]);


In [ ]:
Nu, Nv, Nw = MyDomain.Nu, MyDomain.Nv, MyDomain.Nw
centre = (MyDomain.nodes_u[end]/2, MyDomain.nodes_v[end]/2, MyDomain.nodes_w[end]/2)

radial(i, j, k) = hypot(MyDomain.nodes_u[i] - centre[1],
                        MyDomain.nodes_v[j] - centre[2],
                        MyDomain.nodes_w[k] - centre[3])

Φ_analytical = vec([1.602e-19 / (4π * FITToolbox.ε₀ * max(radial(i,j,k), 0.05))
                    for i = 1:Nu, j = 1:Nv, k = 1:Nw])

# Sample the u–v diagonal of the centre z-plane, from the centre outward.
i_c = findlast(≤(centre[1]), MyDomain.nodes_u)
k_c = findlast(≤(centre[3]), MyDomain.nodes_w)
idx = i_c-1:-1:1

line = [1 + (i-1) + (i-1)*Nu + (k_c-1)*Nu*Nv for i in idx]
r    = [radial(i, i, k_c)                    for i in idx]

dof = round(Int, 100 * (Enlargedomain.Np / MyDomain.Np - 1))

fig = Figure(size = (800, 600))
ax = Axis(fig[1, 1], xlabel = "Distance from Center (m)", ylabel = "Potential Φ (V)",
          title = "Potential Comparison on Grid Nodes", yscale = log10)
lines!(ax, r, Φ_analytical[line];      label = "Analytical", linestyle = :dash, color = :black, linewidth = 2)
lines!(ax, r, Φ_Robin[line];           label = "Robin (+0% DoF)",              color = :blue,   linewidth = 2)
lines!(ax, r, Φ_Robin_Enlarged[line];  label = "Robin enlarged (+$(dof)% DoF)", color = :orange, linewidth = 2)
axislegend(ax, position = :rt)
fig